<a href="https://colab.research.google.com/github/KErarslan/sqk-spatial-kernels/blob/main/SpatialQuantumKernels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Distance-Encoded Quantum Kernels for Spatial Estimation
**Erarslan, K. (2026)** — *Computers & Geosciences* (under review)  
DOI: 10.5281/zenodo.21461418 | GitHub: https://github.com/KErarslan/sqk-spatial-kernels

## Contents
| # | Section | Mode |
|---|---------|------|
| 1 | Setup & imports | SIM + IBM |
| 2 | Kalgoorlie data & classical reference K_V | SIM |
| 3 | Core functions (LOOCV, Statevector kernel) | SIM |
| 4 | Standard feature maps (ZZFeatureMap, PauliFeatureMap) → Table 1 | SIM |
| 5 | Scalar distance kernel SK_rx → Table 2 | SIM |
| 6 | IBM credentials (required for Sections 7 & 9) | IBM |
| 7 | SK_rx on IBM hardware (ibm_marrakesh, ibm_fez, ibm_kingston) | IBM |
| 8 | Anisotropic scalar kernel ASK — h_equiv (Eq. 6-7) | SIM |
| 9 | ASK on IBM hardware — three experiments (α=1.7, 2.5, 0.5) | IBM |
|10 | Complete results Table 3 + Figure 7 | SIM/IBM |
|11 | Block model — Table 4, Figures 8-9 | SIM |
|12 | Retrieve any job result by ID | IBM |

> **SIM** cells run without IBM Quantum access.  
> **IBM** cells require credentials in Section 6.
> All hardware results can be verified via job IDs in `ibm_quantum_job_ids.txt`.


In [ ]:
# Section 1 — Setup
# [SIM + IBM] Run once to install dependencies
!pip install qiskit qiskit-ibm-runtime numpy scipy scikit-learn matplotlib -q
print('Installation complete')


In [ ]:
# [SIM] Core imports
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.linalg import solve
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, PauliFeatureMap
from qiskit.quantum_info import Statevector
import warnings; warnings.filterwarnings('ignore')
print('Imports OK')


## Section 2 — Data & Classical Reference

In [ ]:
# [SIM] Kalgoorlie Northern Zone — n=20 RC drillholes
# Coordinates: [Easting (m), Northing (m), Elevation (m)]
# Au grades: log-transformed
# Source: Riversgold Ltd. (used with permission)

coords = np.array([
    [4120.40,8110.20,162.40],[4130.10,8141.40,162.20],
    [4140.80,8128.80,162.10],[4151.00,8133.10,162.00],
    [4102.10,8109.80,161.90],[4141.20,8157.10,161.70],
    [4130.90,8136.10,161.60],[4151.00,8110.60,161.50],
    [4101.50,8145.80,161.30],[4111.30,8130.90,161.10],
    [4122.10,8120.40,162.30],[4113.40,8151.20,162.20],
    [4142.80,8133.30,162.10],[4153.10,8151.60,161.90],
    [4153.60,8152.10,161.80],[4142.90,8122.70,161.60],
    [4123.40,8130.30,161.50],[4133.80,8151.80,161.30],
    [4103.20,8140.10,161.20],[4114.50,8133.40,161.00],
])
au_log = np.log(np.array([
    2.010,1.250,1.100,0.745,1.920,0.630,1.205,1.380,
    1.565,1.470,1.880,1.545,0.880,1.185,1.170,1.390,
    1.610,0.450,1.630,1.910]))
n = len(coords)

# Feature scaling: koordinatları [0, π] aralığına
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, np.pi))
X = scaler.fit_transform(coords)  # shape: (20, 3)

print(f"n = {n} | Feature dims = {X.shape[1]}")
print(f"X range: [{X.min():.3f}, {X.max():.3f}]")



In [ ]:
# [SIM] Classical covariance reference K_V
# Spherical model fitted by weighted least squares (Fig. 2)
# Parameters: C0=0.015, C=0.206, a_min=28.1m (0°), a_max=66.0m (90°), f_V=0.4
# Rotation-based formulation (Eq. 6-7) — see h_equiv_v3 in Section 8

# 3D-VSK LMC-Ellipsoidal kovaryans matrisi
B_NUG = 0.015; B_STR = 0.206
A_MIN = 28.1;  A_MAX = 66.0; T_MAX = 90.0

def a_ell(az, amin=A_MIN, amax=A_MAX, tmax=T_MAX):
    t = np.radians(az % 180); tm = np.radians(tmax)
    return amin + (amax - amin) * np.cos(t - tm)**2

def cov_lmc(xi, xj):
    """3D-VSK LMC-Ellipsoidal covariance"""
    if np.allclose(xi[:2], xj[:2]): return B_NUG + B_STR
    dx=xj[0]-xi[0]; dy=xj[1]-xi[1]; dz=xj[2]-xi[2]
    hh=np.sqrt(dx**2+dy**2)
    az=np.degrees(np.arctan2(dy,dx))%180
    ah=a_ell(az); av=ah*0.4
    he=np.sqrt((hh/ah)**2+(dz/av)**2)*ah if ah>0 else abs(dz)
    if he>=ah: return 0.
    return B_STR*(1-(1.5*he/ah-0.5*(he/ah)**3))

KV = np.array([[cov_lmc(coords[i],coords[j])
                for j in range(n)] for i in range(n)])
# Normalize to [0,1] diagonal
KV_norm = KV / (B_NUG + B_STR)

lmin_kv = np.linalg.eigvalsh(KV).min()
print(f"\nK_V (3D-VSK):")
print(f"  λ_min = {lmin_kv:.4f}  (PSD: {lmin_kv >= -1e-10})")
print(f"  ||K_V||_F = {np.linalg.norm(KV,'fro'):.4f}")



## Section 3 — Core Functions

In [ ]:
# [SIM] Leave-one-out cross-validation with quantum kernel
# K_Q replaces K_V in the ordinary kriging system (Eq. 1-3)

def sqkk_loocv(KQ, Z):
    """
    Leave-one-out cross-validation with quantum kernel K_Q.
    K_Q replaces K_V in the ordinary kriging system.
    """
    n = len(Z)
    preds = np.zeros(n); vars_ = np.zeros(n)

    for i in range(n):
        # Train indices (all except i)
        tr = [j for j in range(n) if j != i]
        m = len(tr)

        # Kriging matrix (m+1 × m+1) with Lagrange
        K = np.zeros((m+1, m+1))
        for a in range(m):
            K[a,a] = KQ[tr[a], tr[a]]
            for b in range(a+1, m):
                K[a,b] = KQ[tr[a], tr[b]]
                K[b,a] = K[a,b]
        K[:m, m] = 1.; K[m, :m] = 1.

        # RHS
        k0 = np.array([KQ[i, tr[a]] for a in range(m)] + [1.])

        try:
            w = solve(K, k0, assume_a='sym')
        except:
            w = np.linalg.lstsq(K, k0, rcond=None)[0]

        preds[i] = float(np.dot(w[:m], Z[tr]))
        vars_[i] = max(0., float(KQ[i,i] - np.dot(w, k0)))

    res = Z - preds
    r2 = 1 - np.sum(res**2)/np.sum((Z-Z.mean())**2)
    rmse = np.sqrt(np.mean(res**2))
    me = np.mean(res)
    msdr = np.mean(res**2 / (vars_ + 1e-10))
    return {'R2':float(r2), 'RMSE':float(rmse),
            'ME':float(me), 'MSDR':float(msdr),
            'pred':preds, 'var':vars_, 'res':res}

print("\nH4: SQKK LOOCV")
print("="*45)
results_h4 = {}
for name, KQ in KQ_results.items():
    r = sqkk_loocv(KQ, au_log)
    results_h4[name] = r
    print(f"\n  [{name}]")
    print(f"  R²   = {r['R2']:.3f}  (H4: {'✓' if r['R2'] > 0 else '✗'})")
    print(f"  RMSE = {r['RMSE']:.3f}")
    print(f"  ME   = {r['ME']:+.3f}")
    print(f"  MSDR = {r['MSDR']:.3f}")

# K_V referans LOOCV
def kv_loocv(KV, Z):
    return sqkk_loocv(KV/(B_NUG+B_STR), Z)

r_kv = kv_loocv(KV, au_log)
print(f"\n  [K_V 3D-VSK — referans]")
print(f"  R²   = {r_kv['R2']:.3f}")
print(f"  RMSE = {r_kv['RMSE']:.3f}")



In [ ]:
# [SIM] Quantum kernel computation via Statevector (noiseless simulation)
# K_Q(x_i, x_j) = |<0|U†(x_j) U(x_i)|0>|² = return probability to |0>

# Qiskit 1.x uyumlu — Sampler yerine Statevector kullanıyor

from qiskit import QuantumCircuit
from qiskit.circuit.library import ZZFeatureMap, PauliFeatureMap
from qiskit.quantum_info import Statevector

def compute_kernel_entry(x_i, x_j, feature_map):
    """
    K_Q(x_i, x_j) = |<0|U†(x_i) U(x_j)|0>|²
    Statevector ile tam hesaplama — shots gerektirmez.
    """
    n_qubits = feature_map.num_qubits
    fm_j = feature_map.assign_parameters(x_j)
    fm_i = feature_map.assign_parameters(x_i)
    fm_i_inv = fm_i.inverse()

    qc = QuantumCircuit(n_qubits)
    qc.compose(fm_j, inplace=True)
    qc.compose(fm_i_inv, inplace=True)

    sv = Statevector(qc)
    prob = abs(sv[0])**2   # |000⟩ amplitude
    return float(prob.real)

def compute_kernel_matrix(X, feature_map, verbose=True):
    """n×n kernel matrisini hesapla."""
    n = len(X)
    K = np.zeros((n, n))
    total = n*(n+1)//2
    count = 0
    for i in range(n):
        K[i,i] = 1.0
        for j in range(i+1, n):
            val = compute_kernel_entry(X[i], X[j], feature_map)
            K[i,j] = val; K[j,i] = val
            count += 1
            if verbose and count % 30 == 0:
                print(f"  {count}/{total} ({100*count/total:.0f}%)")
    return K

print("Fonksiyonlar tanımlandı (Qiskit 1.x uyumlu).")

## Section 4 — Standard Feature Maps (ZZFeatureMap, PauliFeatureMap) → Table 1
H1 hypothesis: K_Q shows spatial decay (r < 0, p < 0.05).  
Result: H1 fails for all standard feature maps — R² < 0 in all kriging experiments.

In [ ]:
# [SIM] Feature map definitions — 3 qubits (X, Y, Z coordinates)

n_qubits = X.shape[1]  # 3 qubit (X, Y, Z)

feature_maps = {
    'ZZFeatureMap_d1': ZZFeatureMap(
        feature_dimension=n_qubits,
        reps=1,
        entanglement='linear'
    ),
    'ZZFeatureMap_d2': ZZFeatureMap(
        feature_dimension=n_qubits,
        reps=2,
        entanglement='linear'
    ),
    'PauliFeatureMap_ZZZ': PauliFeatureMap(
        feature_dimension=n_qubits,
        reps=1,
        paulis=['Z','ZZ']
    ),
}

for name, fm in feature_maps.items():
    print(f"{name}: {fm.num_qubits} qubit, depth≈{fm.decompose().depth()}")



In [ ]:
# [SIM] Compute kernel matrices (noiseless Statevector simulation)

print("Simülatör çalıştırılıyor...")
KQ_results = {}

for name, fm in feature_maps.items():
    print(f"\n[{name}]")
   #KQ = compute_kernel_matrix(X, fm, shots=8192)
    KQ = compute_kernel_matrix(X, fm, verbose=True)
    KQ_results[name] = KQ
    eigs = np.linalg.eigvalsh(KQ)
    lmin = eigs.min()
    # Frobenius distance (KV normalized to [0,1])
    KV_n = KV_norm
    dF = np.linalg.norm(KQ - KV_n,'fro') / np.linalg.norm(KV_n,'fro')
    rho = np.sum(KQ * KV_n) / (np.linalg.norm(KQ,'fro') *
                                np.linalg.norm(KV_n,'fro'))
    print(f"  λ_min     = {lmin:.4f}  (H2: {'✓' if lmin >= -1e-10 else '✗'})")
    print(f"  d_F       = {dF:.4f}   (H3: {'✓' if dF < 1 else '✗'})")
    print(f"  ρ(cosine) = {rho:.4f}")



In [ ]:
# [SIM] Table 1: Standard feature map results (Kalgoorlie, n=20)
# Key finding: H2 (PSD) and H3 (structure) pass, but H1 and H4 fail.

print("\n" + "="*60)
print("PHASE 2 RESULTS SUMMARY")
print("="*60)
print(f"{'Method':<28} {'λ_min':>8} {'d_F':>7} {'ρ':>7} {'R²':>7} {'H1':>4} {'H2':>4} {'H3':>4} {'H4':>4}")
print("-"*60)

for name, KQ in KQ_results.items():
    eigs = np.linalg.eigvalsh(KQ)
    lmin = eigs.min()
    KV_n = KV_norm
    dF = np.linalg.norm(KQ-KV_n,'fro')/np.linalg.norm(KV_n,'fro')
    rho = np.sum(KQ*KV_n)/(np.linalg.norm(KQ,'fro')*
                             np.linalg.norm(KV_n,'fro'))
    r2 = results_h4[name]['R2']

    dists=[]; kq_vals=[]
    for i in range(n):
        for j in range(i+1,n):
            dists.append(np.linalg.norm(coords[i]-coords[j]))
            kq_vals.append(KQ[i,j])
    r_corr, p_corr = stats.pearsonr(dists, kq_vals)
    h1 = '✓' if p_corr<0.05 and r_corr<0 else '✗'
    h2 = '✓' if lmin >= -1e-10 else '✗'
    h3 = '✓' if dF < 1 else '✗'
    h4 = '✓' if r2 > 0 else '✗'

    short = name.replace('FeatureMap','FM')
    print(f"{short:<28} {lmin:>8.4f} {dF:>7.3f} {rho:>7.3f} "
          f"{r2:>7.3f} {h1:>4} {h2:>4} {h3:>4} {h4:>4}")

print("-"*60)
print(f"{'K_V (3D-VSK reference)':<28} {lmin_kv:>8.4f} {'—':>7} {'—':>7} "
      f"{r_kv['R2']:>7.3f} {'—':>4} {'✓':>4} {'—':>4} {'✓':>4}")
print("="*60)



## Section 5 — Scalar Distance Kernel SK_rx (Simulation) → Table 2
Encodes Euclidean distance as Rx rotation angle on 3 qubits.  
K_SK(d) = cos⁶(πd / 2·d_max) — PSD by Proposition 1.

In [ ]:
# [SIM] SK_rx kernel functions
# Scalar distance encoding: theta = (d / d_max) * pi
# K(d) = |<0|Rx(theta)|0>|^2 = cos^2(theta/2) per qubit => cos^6(theta/2) for 3 qubits

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from scipy import stats
import numpy as np

all_dists = [np.linalg.norm(coords[i]-coords[j])
             for i in range(n) for j in range(i+1,n)]
DMAX = max(all_dists)

def scalar_kernel(d, mode='rx'):
    """
    Scalar mesafeyi 3 qubit'e encode et.
    d: mesafe (scalar)
    mode: encoding stratejisi
    """
    theta = (d / DMAX) * np.pi  # [0, π]
    qc = QuantumCircuit(3)

    if mode == 'rx':
        # Rx rotasyonu — basit, monoton
        qc.rx(theta, 0)
        qc.rx(theta, 1)
        qc.rx(theta, 2)

    elif mode == 'rx_cx':
        # Rx + entanglement
        qc.rx(theta, 0)
        qc.rx(theta, 1)
        qc.rx(theta, 2)
        qc.cx(0, 1); qc.cx(1, 2)

    elif mode == 'ry':
        # Ry rotasyonu
        qc.ry(theta, 0)
        qc.ry(theta, 1)
        qc.ry(theta, 2)

    elif mode == 'rz_h':
        # H + Rz
        qc.h([0,1,2])
        qc.rz(theta, 0)
        qc.rz(theta, 1)
        qc.rz(theta, 2)

    sv = Statevector(qc)
    return float(abs(sv[0])**2)

# Önce teorik davranışı kontrol et
print("Teorik K(d) davranışı:\n")
d_test = np.linspace(0, DMAX, 20)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(16,4))
modes = ['rx', 'rx_cx', 'ry', 'rz_h']

for ax, mode in zip(axes, modes):
    k_vals = [scalar_kernel(d, mode) for d in d_test]
    ax.plot(d_test, k_vals, 'o-', lw=2, color='#2980b9')
    ax.axhline(0, color='k', lw=0.8, ls=':')
    ax.set_xlabel('Distance (m)'); ax.set_ylabel('K(d)')
    ax.set_title(f'mode={mode}\nK(0)={k_vals[0]:.3f}, K(max)={k_vals[-1]:.3f}')
    ax.grid(True, alpha=0.3)
    # Monoton azalıyor mu?
    mono = all(k_vals[i] >= k_vals[i+1] for i in range(len(k_vals)-1))
    ax.set_facecolor('#eafaf1' if mono else '#fdf2f8')
    print(f"  {mode}: K(0)={k_vals[0]:.3f}, "
          f"K(max)={k_vals[-1]:.3f}, monoton={mono}")

plt.suptitle("Scalar Distance Encoding — K(d) vs d",
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('scalar_kernel_decay.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nYeşil arka plan = monoton azalıyor = H1 adayı")

In [ ]:
# [SIM] Compute SK_rx kernel matrix and evaluate all hypotheses

def compute_scalar_kernel_matrix(coords_raw, mode, verbose=True):
    n = len(coords_raw)
    K = np.zeros((n, n))
    total = n*(n+1)//2; count = 0
    for i in range(n):
        K[i,i] = 1.0
        for j in range(i+1, n):
            d = np.linalg.norm(coords_raw[i] - coords_raw[j])
            val = scalar_kernel(d, mode)
            K[i,j] = val; K[j,i] = val
            count += 1
            if verbose and count % 30 == 0:
                print(f"  {count}/{total} ({100*count/total:.0f}%)")
    return K

print("Scalar Kernel Matrix hesaplanıyor...\n")
SK_results = {}

for mode in ['rx', 'rx_cx', 'ry']:
    print(f"[mode={mode}]")
    KS = compute_scalar_kernel_matrix(coords, mode)
    SK_results[mode] = KS

    lmin = np.linalg.eigvalsh(KS).min()
    dF   = np.linalg.norm(KS-KV_norm,'fro')/np.linalg.norm(KV_norm,'fro')
    rho  = np.sum(KS*KV_norm)/(np.linalg.norm(KS,'fro')*
                                 np.linalg.norm(KV_norm,'fro'))
    r2   = sqkk_loocv(KS, au_log)['R2']
    rmse = sqkk_loocv(KS, au_log)['RMSE']
    dists_= [np.linalg.norm(coords[i]-coords[j])
              for i in range(n) for j in range(i+1,n)]
    ks_v  = [KS[i,j] for i in range(n) for j in range(i+1,n)]
    r_c, p_c = stats.pearsonr(dists_, ks_v)

    print(f"  K range: [{np.array(ks_v).min():.4f}, {np.array(ks_v).max():.4f}]")
    print(f"  λ_min   = {lmin:.4f}  H2: {'✓' if lmin>=-1e-10 else '✗'}")
    print(f"  d_F     = {dF:.4f}  H3: {'✓' if dF<1 else '✗'}")
    print(f"  ρ       = {rho:.4f}")
    print(f"  r(dist) = {r_c:.4f}, p={p_c:.4f}  H1: {'✓' if p_c<0.05 and r_c<0 else '✗'}")
    print(f"  R²      = {r2:.4f}  H4: {'✓' if r2>0 else '✗'}")
    print(f"  RMSE    = {rmse:.4f}\n")

# Özet tablo
print("="*60)
print(f"{'Method':<12} {'λ_min':>7} {'d_F':>6} {'ρ':>6} {'R²':>7}  H1 H2 H3 H4")
print("-"*60)
for mode, KS in SK_results.items():
    lmin = np.linalg.eigvalsh(KS).min()
    dF   = np.linalg.norm(KS-KV_norm,'fro')/np.linalg.norm(KV_norm,'fro')
    rho  = np.sum(KS*KV_norm)/(np.linalg.norm(KS,'fro')*
                                 np.linalg.norm(KV_norm,'fro'))
    r2   = sqkk_loocv(KS, au_log)['R2']
    dists_= [np.linalg.norm(coords[i]-coords[j])
              for i in range(n) for j in range(i+1,n)]
    ks_v  = [KS[i,j] for i in range(n) for j in range(i+1,n)]
    r_c,p_c = stats.pearsonr(dists_,ks_v)
    h1='✓' if p_c<0.05 and r_c<0 else '✗'
    h2='✓' if lmin>=-1e-10 else '✗'
    h3='✓' if dF<1 else '✗'
    h4='✓' if r2>0 else '✗'
    print(f"{'SK_'+mode:<12} {lmin:>7.4f} {dF:>6.3f} {rho:>6.3f} {r2:>7.3f}   {h1}  {h2}  {h3}  {h4}")
print("-"*60)
print(f"{'K_V 3D-VSK':<12} {'0.0195':>7} {'—':>6} {'—':>6} {'0.735':>7}   —  ✓  —  ✓")
print("="*60)


## Section 6 — IBM Quantum Credentials (required for Sections 7 & 9)

In [ ]:
# [IBM] Set credentials — required only for hardware experiments
# Option A (Google Colab): add IBM_TOKEN to Colab Secrets (key icon, left sidebar)
# Option B (Local / other): replace 'YOUR_IBM_TOKEN_HERE' below
#
# IBM Quantum Open Plan: https://quantum.ibm.com
# Free account required; ibm_kingston used in the paper (156 qubits, Eagle r3)

try:
    from google.colab import userdata
    API_KEY = userdata.get('IBM_TOKEN')
except Exception:
    API_KEY = 'YOUR_IBM_TOKEN_HERE'

BACKEND_NAME = 'ibm_kingston'   # backend used in the paper
print('Credentials loaded. Token hidden for security.')


In [ ]:
# [IBM] Connect to IBM Quantum service
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel='ibm_quantum', token=API_KEY)
backend = service.backend(BACKEND_NAME)
print(f'Connected: {backend.name} | Queue: {backend.status().pending_jobs} jobs')

# Transpile pass manager
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)


## Section 7 — SK_rx on IBM Hardware
Three backends: ibm_marrakesh, ibm_fez (1,024 shots), ibm_kingston (8,192 shots).  
Key finding: despite spatial structure preserved (ρ≈0.945), PSD margin λ_min=+0.0001 fails on all backends.

In [ ]:
# [IBM] Build SK_rx circuits for hardware
# theta = (d / d_max) * pi; 3 qubits; n(n+1)/2 = 210 circuits

SHOTS_SK = 1024
DMAX = max(np.linalg.norm(coords[i]-coords[j])
           for i in range(n) for j in range(i+1,n))

def build_rx_circuit(d, dmax):
    theta = (d / dmax) * np.pi
    qc = QuantumCircuit(3)
    for q in range(3): qc.rx(theta, q)
    qc.measure_all()
    return qc

circuits_rx, idx_rx = [], []
for i in range(n):
    for j in range(i, n):
        d = np.linalg.norm(coords[i]-coords[j])
        circuits_rx.append(pm.run(build_rx_circuit(d, DMAX)))
        idx_rx.append((i, j))
print(f'{len(circuits_rx)} circuits ready')


In [ ]:
# [IBM] Run SK_rx — ibm_kingston (8,192 shots)
# Job ID: d8ntk9j2d42s73ceb8l0

backend_k = service.backend('ibm_kingston')
pm_k = generate_preset_pass_manager(optimization_level=1, backend=backend_k)
circuits_rx_k = [pm_k.run(build_rx_circuit(
    np.linalg.norm(coords[i]-coords[j]), DMAX))
    for i in range(n) for j in range(i,n)]

sampler_k = SamplerV2(backend_k)
job_sk_k = sampler_k.run(circuits_rx_k, shots=8192)
print(f'SK_rx ibm_kingston job submitted: {job_sk_k.job_id()}')
print('Expected job ID: d8ntk9j2d42s73ceb8l0')


## Section 8 — Anisotropic Scalar Kernel (ASK) — Simulation
Rotation-based h_e formulation (Eq. 6-7, Proposition 2).  
h_equiv_v3 rotates the separation vector onto major/minor anisotropy axes — this is the CORRECTED formula (the earlier formula cancelled θ algebraically).

In [ ]:
# [SIM] h_equiv_v3 — corrected rotation-based lag-distance (Eq. 6-7)
# Projects separation vector onto major/minor axes, scales each by its range.
# This is the formula used in all ASK simulations and hardware experiments.

def h_equiv_v3(xi, xj, a_min, a_max, theta_max_deg, f_v=None):
    """
    Ellipsoidal equivalent lag distance (Eq. 6-7).
    Rotates (dx,dy) onto major/minor anisotropy axes, then scales.
    a_min, a_max: fitted variogram ranges (m)
    theta_max_deg: azimuth of major range (degrees)
    f_v: vertical anisotropy factor (None = 2D)
    """
    dx = xj[0]-xi[0]; dy = xj[1]-xi[1]
    tm = np.radians(theta_max_deg)
    u_major =  dx*np.cos(tm) + dy*np.sin(tm)
    u_minor = -dx*np.sin(tm) + dy*np.cos(tm)
    s_minor = a_max / a_min if a_min > 1e-9 else 1.0
    he_sq = u_major**2 + (u_minor * s_minor)**2
    if f_v is not None and len(xi) > 2:
        a_v = a_max * f_v
        dz = xj[2]-xi[2]
        he_sq += (dz * a_max / a_v)**2
    return np.sqrt(he_sq)

def compute_ask_v3(coords, a_min, a_max, theta_max_deg, alpha,
                   f_v=None, nq=3):
    """
    Compute ASK kernel matrix (Eq. 8).
    K_ASK(x_i, x_j) = cos^(2*nq)(alpha * h_e / HE_MAX * pi/2)
    """
    n = len(coords); K = np.eye(n)
    all_he = [h_equiv_v3(coords[i], coords[j], a_min, a_max,
                         theta_max_deg, f_v)
              for i in range(n) for j in range(i+1, n)]
    he_max = max(all_he); idx = 0
    for i in range(n):
        for j in range(i+1, n):
            theta = min(alpha * (all_he[idx]/he_max) * np.pi, np.pi)
            idx += 1
            qc = QuantumCircuit(nq)
            for q in range(nq): qc.rx(theta, q)
            K[i,j] = K[j,i] = float(abs(Statevector(qc)[0])**2)
    return K, he_max

# Kalgoorlie parameters (fitted variogram, Section 3.1)
A_MIN=28.1; A_MAX=66.0; T_MAX=90.0; FV=0.4
print('h_equiv_v3 and compute_ask_v3 defined (Eq. 6-8)')


In [ ]:
# [SIM] Alpha parameter scan — PSD boundary scan protocol (Supplementary S1)
# Find largest alpha* with lambda_min(K_ASK) >= 0

print(f'Alpha scan (Kalgoorlie, ref_scale = d_max = {A_MAX:.1f} m)')
print(f'{"alpha":>6} {"lambda_min":>11} {"rho":>7} {"r(h_e)":>8} {"R2":>8}')
print('-'*48)

best_K = None; best_alpha = None; best_r2 = -999

for alpha in np.arange(0.5, 3.05, 0.1):
    KA, HE_MAX = compute_ask_v3(coords, A_MIN, A_MAX, T_MAX, alpha, FV)
    lm = np.linalg.eigvalsh(KA).min()
    rho = (np.sum(KA*KV_norm) /
           (np.linalg.norm(KA,'fro')*np.linalg.norm(KV_norm,'fro')))
    dists = [np.linalg.norm(coords[i]-coords[j])
             for i in range(n) for j in range(i+1,n)]
    kvals = [KA[i,j] for i in range(n) for j in range(i+1,n)]
    r, _ = stats.pearsonr(dists, kvals)
    res = sqkk_loocv(KA, au_log)
    psd_ok = '*' if lm >= 0 else ' '
    print(f'{alpha:>6.1f} {lm:>+11.5f}{psd_ok} {rho:>7.3f} {r:>8.3f} {res["R2"]:>8.3f}')
    if lm >= 0 and res['R2'] > best_r2:
        best_r2 = res['R2']; best_alpha = alpha; best_K = KA.copy()

ALPHA_OPT = best_alpha
KA_OPT, HE_MAX = compute_ask_v3(coords, A_MIN, A_MAX, T_MAX, ALPHA_OPT, FV)
print(f'\nSelected alpha* = {ALPHA_OPT:.1f} | R2 = {best_r2:.3f} | HE_MAX = {HE_MAX:.2f} m')


In [ ]:
# [SIM] Bootstrap 95% CI for OK and ASK (B=10,000)
from sklearn.utils import resample

B = 10000
r2_boot_ok = []; r2_boot_ask = []
for seed in range(B):
    idx = resample(range(n), n_samples=n, replace=True, random_state=seed)
    idx = np.array(idx)
    KV_b = KV[np.ix_(idx,idx)]
    KA_b = KA_OPT[np.ix_(idx,idx)]
    z_b = au_log[idx]
    r2_boot_ok.append(sqkk_loocv(KV_b, z_b)['R2'])
    r2_boot_ask.append(sqkk_loocv(KA_b, z_b)['R2'])

ci_ok  = np.percentile(r2_boot_ok,  [2.5, 97.5])
ci_ask = np.percentile(r2_boot_ask, [2.5, 97.5])
print(f'OK  R2 = {np.median(r2_boot_ok):.3f}  95% CI: [{ci_ok[0]:.2f}, {ci_ok[1]:.2f}]')
print(f'ASK R2 = {np.median(r2_boot_ask):.3f}  95% CI: [{ci_ask[0]:.2f}, {ci_ask[1]:.2f}]')


## Section 9 — ASK on IBM Hardware
Three experiments on ibm_kingston (4,096 shots each, July 2026).  

| Experiment | α | Norm. | λ_min | R² | Status |
|---|---|---|---|---|---|
| June 2026 | 1.7 | d_max | +0.0038 | 0.569 | ✓ PSD preserved |
| July 2026 A | 2.5 | d_max | −1.624 | — | ✗ Non-monotone regime (83.7% pairs above π/2) |
| July 2026 B | 0.5 | HE_MAX | +0.0048 | 0.564 | ✓ PSD preserved |

In [ ]:
# [IBM] Build ASK circuit — corrected formula (Eq. 8, Section 6.2)
# theta = alpha * (h_e / ref_scale) * pi
# ref_scale: d_max (compact domain) or HE_MAX (large domain)

def build_ask_circuit(xi, xj, alpha, ref_scale):
    """Build ASK circuit for sample pair (xi, xj)."""
    he = h_equiv_v3(xi, xj, A_MIN, A_MAX, T_MAX, FV)
    theta = min(alpha * (he / ref_scale) * np.pi, np.pi)
    qc = QuantumCircuit(3)
    for q in range(3): qc.rx(theta, q)
    qc.measure_all()
    return qc

def run_ask_hardware(service, backend_name, alpha, ref_scale, shots=4096):
    """Submit ASK kernel evaluation to IBM hardware."""
    backend = service.backend(backend_name)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    circuits = []
    idx_pairs = []
    for i in range(n):
        for j in range(i, n):
            qc = pm.run(build_ask_circuit(coords[i], coords[j], alpha, ref_scale))
            circuits.append(qc); idx_pairs.append((i,j))
    sampler = SamplerV2(backend)
    job = sampler.run(circuits, shots=shots)
    print(f'Job submitted: {job.job_id()}')
    return job, idx_pairs

def parse_ask_result(job, idx_pairs, n):
    """Parse SamplerV2 result into kernel matrix."""
    result = job.result()
    K = np.zeros((n, n))
    for k, (i,j) in enumerate(idx_pairs):
        counts = result[k].data.meas.get_counts()
        total = sum(counts.values())
        K[i,j] = K[j,i] = counts.get('000', 0) / total
    np.fill_diagonal(K, 1.0)
    return K

print('ASK hardware functions defined')


In [ ]:
# [IBM] Experiment 1 — ASK alpha=1.7, d_max normalisation (June 2026)
# Job ID: d8o3r4b2d42s73cejlcg | Result: lambda_min=+0.0038, R2=0.569
# To reproduce:
# job1, idx1 = run_ask_hardware(service, 'ibm_kingston', alpha=1.7, ref_scale=DMAX)
# KA_hw17 = parse_ask_result(job1, idx1, n)

# ── Retrieve stored result ──────────────────────────────────
job1 = service.job('d8o3r4b2d42s73cejlcg')
KA_hw17 = parse_ask_result(job1, [(i,j) for i in range(n) for j in range(i,n)], n)
lm17 = np.linalg.eigvalsh(KA_hw17).min()
r2_17 = sqkk_loocv(KA_hw17, au_log)['R2']
print(f'ASK alpha=1.7 ibm_kingston: lambda_min={lm17:+.4f}, R2={r2_17:.3f}')


In [ ]:
# [IBM] Experiment 2 — ASK alpha=2.5, d_max normalisation (July 2026) — FAILED
# 83.7% of pairs in non-monotone cos^6 regime (angle > pi/2)
# Job ID: d9cc8kk1osis73bjm2ig | Result: lambda_min=-1.624, PSD VIOLATED

job2 = service.job('d9cc8kk1osis73bjm2ig')
KA_hw25 = parse_ask_result(job2, [(i,j) for i in range(n) for j in range(i,n)], n)
lm25 = np.linalg.eigvalsh(KA_hw25).min()
neg25 = (np.linalg.eigvalsh(KA_hw25) < 0).sum()
print(f'ASK alpha=2.5 d_max ibm_kingston: lambda_min={lm25:+.4f}, neg. eig={neg25}')
print(f'Status: {"FAILED — non-monotone regime" if lm25 < 0 else "OK"}')


In [ ]:
# [IBM] Experiment 3 — ASK alpha=0.5, HE_MAX normalisation (July 2026) — SUCCEEDED
# HE_MAX = max h_e over all sample pairs (ensures monotone regime)
# Job ID: d9ce8akjeosc73fg5p40 | Result: lambda_min=+0.0048, R2=0.564

_, HE_MAX_VAL = compute_ask_v3(coords, A_MIN, A_MAX, T_MAX, 0.5, FV)
print(f'HE_MAX = {HE_MAX_VAL:.2f} m')

job3 = service.job('d9ce8akjeosc73fg5p40')
KA_hw05 = parse_ask_result(job3, [(i,j) for i in range(n) for j in range(i,n)], n)
lm05 = np.linalg.eigvalsh(KA_hw05).min()
r2_05 = sqkk_loocv(KA_hw05, au_log)['R2']
print(f'ASK alpha=0.5 HE_MAX ibm_kingston: lambda_min={lm05:+.4f}, R2={r2_05:.3f}')


## Section 10 — Complete Results (Table 3)

In [ ]:
# [SIM] Table 3 — Complete results (simulation + IBM hardware)
# R2(eps): eigenvalue eps-clipping (10^-4) where lambda_min < 0

def eps_clip_r2(K, z, eps=1e-4):
    eigs, vecs = np.linalg.eigh(K)
    K_c = vecs @ np.diag(np.clip(eigs, eps, None)) @ vecs.T
    return sqkk_loocv(K_c, z)['R2']

header = f'{"Method":<26} {"Shots":>6} {"lam_min":>8} {"Neg":>4} {"d_F":>6} {"r(d)":>7} {"R2":>7}'
print(header)
print('-'*70)

KV_norm = KV / np.sqrt(np.outer(np.diag(KV), np.diag(KV)))

rows = [
    ('SK_rx (Sim)',        SK_results['rx'],  np.inf),
    ('SK_rx ibm_kingston', None,              8192),   # load from job
    ('ASK a=1.7 ibm_k',   KA_hw17,           4096),
    ('ASK a=2.5 d_max',   KA_hw25,           4096),
    ('ASK a=0.5 HE_MAX',  KA_hw05,           4096),
    ('ASK a=OPT (Sim)',   KA_OPT,            np.inf),
    ('K_V (classical)',   KV,                np.inf),
]
for label, K, shots in rows:
    if K is None: continue
    eigs = np.linalg.eigvalsh(K)
    lm   = eigs.min()
    neg  = (eigs < 0).sum()
    dF   = np.linalg.norm(K - KV_norm,'fro') / np.linalg.norm(KV_norm,'fro') if neg==0 else '—'
    dsts = [np.linalg.norm(coords[i]-coords[j]) for i in range(n) for j in range(i+1,n)]
    kv   = [K[i,j] for i in range(n) for j in range(i+1,n)]
    r,_  = stats.pearsonr(dsts, kv)
    r2   = sqkk_loocv(K, au_log)['R2'] if neg==0 else '—'
    shots_s = f'{shots:,.0f}' if shots != np.inf else 'inf'
    dF_s = f'{dF:.3f}' if isinstance(dF, float) else dF
    r2_s = f'{r2:.3f}' if isinstance(r2, float) else r2
    print(f'{label:<26} {shots_s:>6} {lm:>+8.4f} {neg:>4} {dF_s:>6} {r:>7.3f} {r2_s:>7}')


## Section 11 — Block Model (Table 4, Figures 8-9)
5×5×1 m blocks within convex hull of drill data.  
Cut-off grades: 0.3, 0.5, 1.0, 2.0 g/t Au.

In [ ]:
# [SIM] Block model construction
from scipy.spatial import ConvexHull

# Grid of 5x5x1 m blocks within convex hull
xy = coords[:, :2]
hull = ConvexHull(xy)
x_range = np.arange(xy[:,0].min(), xy[:,0].max()+5, 5)
y_range = np.arange(xy[:,1].min(), xy[:,1].max()+5, 5)
z_mid   = coords[:,2].mean()  # single elevation layer

block_xyz = []
for x in x_range:
    for y in y_range:
        pt = np.array([x, y])
        # Check if inside convex hull
        in_hull = all((hull.equations[:, :-1] @ pt + hull.equations[:, -1]) <= 0.5)
        if in_hull:
            block_xyz.append([x, y, z_mid])
block_xyz = np.array(block_xyz)
print(f'Blocks in convex hull: {len(block_xyz)}')

# Kriging estimation for each block using K_V and ASK
def krige_blocks(K_train, z_train, coords_train, coords_pred):
    """Kriging prediction at new locations."""
    preds = []
    for xp in coords_pred:
        k0 = np.array([cov_lmc(xi, xp) for xi in coords_train])
        n_t = len(z_train)
        A = np.vstack([np.hstack([K_train, np.ones((n_t,1))]),
                       np.hstack([np.ones((1,n_t)), [[0]]])])
        b = np.append(k0, 1)
        try:
            w = np.linalg.solve(A, b)[:-1]
            preds.append(float(w @ z_train))
        except np.linalg.LinAlgError:
            preds.append(float(np.mean(z_train)))
    return np.array(preds)

# Estimate blocks with OK (K_V) and ASK
block_ok  = krige_blocks(KV, au_log, coords, block_xyz)
block_ask = krige_blocks(KA_OPT, au_log, coords, block_xyz)

# Back-transform from log scale
density = 2.65  # t/m3
block_vol = 5 * 5 * 1  # m3

print('Table 4: Resource estimate comparison')
print(f'{"Cutoff":>7} {"Method":>6} {"Blocks":>7} {"Tonnes(kt)":>10} {"Avg Au":>8} {"Cont. oz":>10}')
print('-'*52)
for cutoff in [0.3, 0.5, 1.0, 2.0]:
    for label, blk in [('OK', block_ok), ('ASK', block_ask)]:
        au_lin = np.exp(blk)  # back-transform
        mask = au_lin >= cutoff
        nb_ = mask.sum()
        tonnes = nb_ * block_vol * density / 1000
        avg = au_lin[mask].mean() if nb_ > 0 else 0
        oz = tonnes * 1000 * avg / 31.1035  # troy oz
        print(f'{cutoff:>7.1f} {label:>6} {nb_:>7} {tonnes:>10.1f} {avg:>8.3f} {oz:>10.1f}')


## Section 12 — Retrieve Any Hardware Result by Job ID
All job IDs are listed in `ibm_quantum_job_ids.txt`.  
Requires IBM credentials from Section 6.

In [ ]:
# [IBM] Template: retrieve and parse any hardware result

JOB_ID = ''  # paste any Job ID from ibm_quantum_job_ids.txt

# Available job IDs (Kalgoorlie experiments):
# SK_rx  | ibm_marrakesh | 1024 shots | d8nsi4rnn5bs738v1rlg
# SK_rx  | ibm_fez       | 1024 shots | d8nt8mjqv2lc7389adug
# SK_rx  | ibm_kingston  | 8192 shots | d8ntk9j2d42s73ceb8l0
# ASK marrakesh (encoding error — see Suppl. S2) | 2048 | d8o3fkjqv2lc7389is20
# ASK alpha=1.7 | ibm_kingston | 4096 | d8o3r4b2d42s73cejlcg
# ASK alpha=2.5 d_max (FAILED) | ibm_kingston | 4096 | d9cc8kk1osis73bjm2ig
# ASK alpha=0.5 HE_MAX (OK)   | ibm_kingston | 4096 | d9ce8akjeosc73fg5p40

if JOB_ID:
    job = service.job(JOB_ID)
    print(f'Status: {job.status()}')
    result = job.result()
    pairs = [(i,j) for i in range(n) for j in range(i,n)]
    K_hw = np.zeros((n,n)); np.fill_diagonal(K_hw, 1.0)
    for k,(i,j) in enumerate(pairs):
        counts = result[k].data.meas.get_counts()
        K_hw[i,j] = K_hw[j,i] = counts.get('000',0)/sum(counts.values())
    eigs = np.linalg.eigvalsh(K_hw)
    r2 = sqkk_loocv(K_hw, au_log)['R2']
    print(f'lambda_min = {eigs.min():+.4f} | neg. eig = {(eigs<0).sum()} | R2 = {r2:.3f}')
else:
    print('Set JOB_ID to a job ID from ibm_quantum_job_ids.txt and re-run.')
